# 04 · Simulation outputs: from TNG snapshots to gas fields

`skewerjax.simulations` turns the outputs of a hydrodynamical simulation (here a TNG snapshot) into the gas along the skewers that the optical-depth integrator needs. Layers:

| layer | module | does | arrays |
|---|---|---|---|
| 1 | `readers` | HDF5 I/O and units (`ArepoHDF5Reader`) | NumPy |
| 2 | `gas_cells` | `GasCells`: positions, volumes, physical fields | pytree |
| 3 | `gas_physics` | n_H, T, x_HI, metal ion fractions (CLOUDY) per cell | JAX |
| 4 | `selection` | the cells that can reach the skewers (k-d tree, streamed) | NumPy |
| 5 | `deposition` | which cell fills which piece of every dual segment (Voronoi, or fake_spectra's top hats) | NumPy + JAX |
| 6 | `absorbers` | pieces → `core.LoSSamples` → `core.optical_depth_from_samples` | JAX |

`SimulationSkewers` runs them for any `ExtractionOptions` / `IntegrationOptions` (defaults: SkewerJAX's). Data:
TNG50-4 snapshot 25 (z = 3.0, 35 cMpc/h, 19.3 M cells), read-only. Runtime: ~1 min. The τ of these skewers is computed in notebook 06.

**Requires:** skewers (03, with their a(x) from 02) and a simulation snapshot; the atomic table (01) for the ion masses.  
**Independent of:** the toy samplers (05) and any `LineBundle`.

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np, jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)          # SkewerJAX follows this flag; it never sets it itself
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import (use_style, ENTITY_COLORS as EC, MODE_STYLES as MS, PALETTE as TOL, FG, LABELS, FIGSIZES,
                                      add_panel_letters, plot_dag, plot_spectra, zoom_inset)
use_style()                                        # the dark, colour-blind-safe style of every figure (skewerjax.utils.plotting)
%matplotlib inline
from skewerjax.core import LineBundle, SkewerBundle, Cosmology, phys
from skewerjax.simulations import ArepoHDF5Reader, SimulationSkewers, ExtractionOptions, IntegrationOptions, HI, Ion, gas_physics as gp
DATA = os.environ.get("SKEWERJAX_DATA", "/home/STORAGE")      # folder with TNG50-4/ (docs/installation.md)
reader = ArepoHDF5Reader(f"{DATA}/TNG50-4", 25)
H = reader.header
print(f"z = {H.redshift:.4f}, a = {H.a:.6f}, box = {H.box_cMpc:.4f} cMpc, {H.n_gas:,} gas cells in {H.n_files} files, "
      f"a H(z) L = {H.vmax_kms:.1f} km/s")

In [ ]:
plot_dag(('simulation',)); plt.show()

## 1 · What the integrator needs, and how the snapshot provides it

For line j of species i, along a path γ (notes `sec_flux.tex`, Eq. tau_straight_los; `core.optical_depth.optical_depth_kernel`
L124–L155):

$$\tau_{j}(v_\dagger)=\mathcal C f_j\int_\gamma \frac{n_{i,\rm com}}{a^{2}}\,\phi_{j}(v_\dagger; T, v_\parallel, a)\,dl_{\rm com}
\;\simeq\;\mathcal C f_j\sum_i\sum_{p\in i}\frac{n_{{\rm com},p}\,\ell_p}{a_p^{2}}\,\phi_{j,p}(v_\dagger),\qquad \sum_{p\in i}\ell_p=w_i .$$

So per piece p of gas on the path the integrator needs the comoving density n_com, the temperature T (the width
b = √(2k_BT/m)), the velocity along the path v_∥ (the centre), the scale factor a (the Hubble flow, from the skewer's
a(x)) and the length ℓ_p. The snapshot provides them in six steps:

| step | code | per cell / piece |
|---|---|---|
| 1. read | `readers.ArepoHDF5Reader.to_gas_cells` L213 | x [cMpc] = `Coordinates` × ckpc/h; ρ [g cm⁻³, physical] = `Density` × unit; u [erg/g] = `InternalEnergy` × (km/s)²; v_pec [km/s] = `Velocities` × √a; X_k = `GFM_Metals`; n_e/n_H; SFR; x_HI (stored) |
| 2. cells | `gas_cells.GasCells` | V = M/ρ (comoving), size V^{1/3} (`size_cMpc` L72) |
| 3. physics | `gas_physics.ion_properties` L322 | n_H = ρ X_H/(A_H m_u) (`element_number_density` L124); T = (γ−1) μ m_p u/k_B, μ = 4/(1+3X+4X n_e), X = 0.76 (`temperature` L129); x_HI stored, star-forming cells recomputed (Rahmati+13 at 10⁴ K, `neutral_fraction` L208); metals f_ion = CLOUDY(n_H, T) (`CloudyIonTable`); **n_ion = ρ X_elem/(A m_u) × f_ion** |
| 4. selection | `selection.select_cells_near_paths` | the cells that can touch the paths (periodic k-d tree, streamed) |
| 5. deposition | `deposition.voronoi_segments` L126 | the pieces of every dual segment: cell, exact length ℓ_p, centroid offset (nearest mesh-generating point = Arepo's own cells) |
| 6. absorbers | `absorbers.absorbers_from_components` L163, `absorber_samples` L237 | s_p, n_p, ℓ_p, T_p, v_∥,p = v_pec·t̂; then n_com = n a_snap³, a(s_p) from the frame → `core.LoSSamples` → τ |

The alternative to step 6 is `SimulationSkewers.point_values` (L280): one value per sampling point, the segment
average n_i = Σ_p n_p ℓ_p / w_i with column-weighted T and v. `fields(species)` wraps them as `SampledField`s (05).

## 2 · Per-cell physics (layer 3)

TNG stores x_HI from its own ionisation equilibrium; SkewerJAX keeps it, recomputing only star-forming cells (cold phase, Rahmati+13).

In [ ]:
cells = reader.to_gas_cells(reader.read_rows(0, ["Coordinates", "Masses", "Density", "InternalEnergy", "ElectronAbundance",
                                                    "NeutralHydrogenAbundance", "GFM_Metals", "StarFormationRate", "Velocities"], stop=400_000))
P_HI, P_Si = gp.ion_properties(cells, HI), gp.ion_properties(cells, Ion("Si", 3))
from skewerjax.utils.plotting import tol_cmap
fig, axs = plt.subplots(1, 2, figsize=(20, 7), layout="constrained")
for ax, P, name in ((axs[0], P_HI, r"$x_{\rm HI}$"), (axs[1], P_Si, r"$f_{\rm SiIII}$")):
    sc = ax.scatter(np.asarray(P["n_H"]), np.asarray(P["T"]), c=np.log10(np.maximum(np.asarray(P["ion_fraction"]), 1e-12)), s=0.4,
                    cmap=tol_cmap("YlOrBr"), vmin=-8, vmax=0, rasterized=True)
    fig.colorbar(sc, ax=ax, label=f"log10 {name}"); ax.set(xscale="log", yscale="log", xlabel=r"$n_{\rm H}$ [cm$^{-3}$]", ylabel="T [K]")
add_panel_letters(axs); fig.suptitle("400,000 TNG50-4 cells: H I (stored equilibrium) and Si III (CLOUDY) fractions", fontsize=22); plt.show()

## 3 · Three grids: cells, sampling points and the spectral grid

Three discretisations meet in a simulated spectrum, and keeping them apart is the key to the deposition (after the
pre-refactor notebook `3_fields_from_simulations`, cells 3–5):
- **the cells:** one value per Voronoi cell (the region closer to its mesh-generating point than to any other);
- **the sampling points of the `SkewerBundle`:** point i stands for its dual segment, between the midpoints with its
  neighbours, of length w_i (the line element);
- **the spectral grid** of v on which τ is evaluated, chosen freely.

A toy two-dimensional mesh with a denser filament in the middle shows all three.

In [ ]:
from skewerjax.utils.plotting import plot_voronoi_discretisations
fig, axs, col = plot_voronoi_discretisations()
plt.show()
print(f"column along the path: exact {col['exact']:.4f} | segment averages x w_i {col['segments']:.4f} "
      f"({col['segments'] / col['exact'] - 1:+.1e}) | point values x w_i {col['points']:.4f} ({col['points'] / col['exact'] - 1:+.1%})")

*Reading the figure.* (a) The path crosses cells of every size; the sampling points do not know about them.
(b) Along the path the density is piecewise constant (white). The value at a point (red diamonds) is whichever cell the
point falls in, so the column Σ n(x_i) w_i depends on the spacing; the segment average (green bars) is the exact
integral of the gas over the segment, so Σ n̄_i w_i is the column for any spacing. (c) The spectral grid is independent
of both: every piece of gas adds its own profile, centred where its a(x) and v_∥ place it. The rest of the notebook
does exactly this on the real snapshot.

## 4 · Skewers and deposition (layers 4–5)

Any `SkewerBundle` can be used (`SimulationSkewers.from_bundle`, with its own a(x)); here three straight skewers along
x, sampled every 0.25 km/s of Hubble flow. Each point stands for its dual segment, and the deposition finds the exact
pieces of the Voronoi cells inside it, so the column is conserved for any spacing.

In [ ]:
cosmo = Cosmology(H0=100 * H.h * 1e6 * phys.Gyr_to_s / phys.Gpc_to_m, Omega_r_0=0.0, Omega_m_0=H.omega_m, Omega_k_0=0.0, Omega_lambda_0=H.omega_l)
L = H.box_cMpc
x = np.linspace(0.0, L, int(round(H.vmax_kms / 0.25)) + 1)
points = np.stack([np.stack([x, np.full_like(x, y), np.full_like(x, z)], -1) for y, z in [(10.0, 30.0), (25.0, 5.0), (40.0, 45.0)]])
bundle = SkewerBundle(spatial_points=jnp.asarray(points), a0=H.a, ICs_defined_at_index=0, ICs_at_emission_or_absorption="absorption", cosmologies=cosmo)
sk = SimulationSkewers.from_bundle(reader, bundle)
print(f"{len(sk.cells):,} cells selected in {sk.selection_s:.1f} s")
seg = sk.segments("voronoi"); A = sk.absorbers(HI)
win = (A.s[0] > 30) & (A.s[0] < 31) & (A.n[0] > 0)
pv = sk.point_values(HI)
fig, axs = plt.subplots(2, 1, figsize=(14, 7), sharex=True, layout="constrained")
axs[0].semilogy(sk.s[0], pv["n"][0], color=FG, lw=1, label=r"segment average $\sum_p n_p\ell_p / w_i$")
axs[0].semilogy(A.s[0][win], A.n[0][win], ".", color=EC["density"], ms=4, label="pieces, at their centroids")
axs[1].semilogy(A.s[0][win], A.T[0][win], ".", color=EC["temperature"], ms=4)
axs[0].set(xlim=(30, 31), ylabel=r"$n_{\rm HI}$ [cm$^{-3}$]"); axs[1].set(xlabel="path length s [cMpc]", ylabel="T [K]"); axs[0].legend()
add_panel_letters(axs); fig.suptitle("Voronoi deposition: piecewise-constant gas, one value per cell crossing", fontsize=22); plt.show()

## 5 · The column does not depend on the sampling

The same three skewers sampled 0.25, 2, 8 and 32 km/s apart (in Hubble flow), with the cells already selected. The
Voronoi deposition integrates every dual segment exactly, so the total N_HI of each skewer is the same at every
spacing. The point deposition (`deposition="points"`: one value per point, the top-hat spheres containing it, no
line element; overlapping spheres are summed) samples the gas instead: its column is off even at 0.25 km/s and drifts
as the points thin out.

In [ ]:
rows = []
for dv_pt in (0.25, 2.0, 8.0, 32.0):
    xs = np.linspace(0.0, L, int(round(H.vmax_kms / dv_pt)) + 1)
    pts = np.stack([np.stack([xs, np.full_like(xs, p[0, 1]), np.full_like(xs, p[0, 2])], -1) for p in points])
    skc = SimulationSkewers(reader, pts, cells=sk.cells)
    N = {kind: skc.column_per_pixel(HI, ExtractionOptions(deposition=kind), nbins=512).sum(-1) for kind in ("voronoi", "points")}
    rows.append((dv_pt, xs.size, N["voronoi"], N["points"]))
N0 = rows[0][2]
fig, ax = plt.subplots(figsize=FIGSIZES["single"], layout="constrained")
for k in range(3):
    c = list(TOL.values())[k]
    ax.plot([r[0] for r in rows], [r[2][k] / N0[k] for r in rows], "o-", color=c, label=f"skewer {k}, Voronoi segments")
    ax.plot([r[0] for r in rows], [r[3][k] / N0[k] for r in rows], "s--", color=c, label=f"skewer {k}, point values")
ax.set(xscale="log", yscale="log", xlabel="spacing of the sampling points [km/s of Hubble flow]",
       ylabel=r"$N_{\rm HI}$ / Voronoi $N_{\rm HI}$ at 0.25 km/s")
ax.axhline(1, color=FG, lw=1); ax.legend(fontsize=11, ncol=2)
ax.set_title("Total H I column per skewer: exact for Voronoi segments at any spacing, not for point values", fontsize=18); plt.show()
print(f"Voronoi: max |N / N(0.25 km/s) - 1| over spacings and skewers = {max(np.abs(r[2] / N0 - 1).max() for r in rows):.1e}")
for dv_pt, n, Nv, Np in rows:
    print(f"spacing {dv_pt:5.2f} km/s ({n:6d} points): log10 N_HI Voronoi {np.round(np.log10(Nv), 6)} | points {np.round(np.log10(Np), 4)}")

**Options.** `ExtractionOptions`: `deposition` ("voronoi" | "tophat" | "points"), `mass_unit` ("amu" | "proton"),
`T_constants`, `b_constants` ("codata" | "fake_spectra"), `x_HI_model` ("stored_sf" | "stored" | "fake_sf" |
"rahmati_all"), `cloudy_density` ("hydrogen" | "fake_spectra"), `cloudy_interpolation` ("linear" | "cubic").
`IntegrationOptions`: `frame` ("exact" | "snapshot"), `normalisation`, `detuning` ("exact" | "linear"),
`periodic`. The non-default values reproduce fake_spectra (notebook 08). Checks:
`scripts/validation/check_tng_fields.py`, `check_tng_optical_depth.py`.

## Summary

- Six layers go from the HDF5 snapshot to the gas along the skewers: reader, cells, physics, selection, deposition,
  absorbers. `SimulationSkewers` runs them for any options.
- Cells, sampling points and the spectral grid are three different grids. The Voronoi deposition keeps the exact
  column of every dual segment for any spacing of the points.
- `SimulationSkewers.fields(species)` gives the same gas as a `core.Fields` of `SampledField`s (05); τ is notebook 06.
- Checks: `scripts/validation/check_tng_fields.py`, `check_tng_optical_depth.py`.